In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

# Load scaled dataset"
df = pd.read_excel(
    "Final_INDOFLOODS_scaled_dataset.xlsx",
    sheet_name=0,
    engine="openpyxl"
)

# --------------------------------------------------
# Population Density
# --------------------------------------------------
df["population_density"] = (
    df["population_within_5km"] / (np.pi * 5**2)
)

# Normalize population density to 0–1
df["population_density"] = (
    (df["population_density"] - df["population_density"].min())
    / (df["population_density"].max() - df["population_density"].min())
)

# --------------------------------------------------
# Scores
# --------------------------------------------------

# Weather Score
df["weather_score"] = (
    0.5 * df["Rainfall_mm"]
    + 0.3 * df["WindSpeed_kmh"]
    + 0.2 * df["Humidity_percent"]
)

# Infrastructure Score
df["infrastructure_score"] = (
    0.5 * df["hospital_count_5km"]
    + 0.3 * df["relief_shelter_count_5km"]
    + 0.2 * df["school_count_5km"]
)

# Accessibility Score
df["accessibility_score"] = (
    0.5 * df["road_density"]
    + 0.3 * df["road_connectivity"]
    + 0.2 * (1 - df["highway_distance"])
)

# Vulnerability Score
df["vulnerability_score"] = (
    0.5 * df["population_density"]
    + 0.3 * (1 - df["infrastructure_score"])
    + 0.2 * (1 - df["accessibility_score"])
)

# Impact Score
df["impact_score"] = (
    0.4 * df["Peak Flood Level (m)"]
    + 0.3 * df["Rainfall_mm"]
    + 0.3 * df["population_density"]
)

# --------------------------------------------------
# Features and Target
# --------------------------------------------------

feature_columns = [
    "Peak Flood Level (m)",
    "weather_score",
    "population_density",
    "infrastructure_score",
    "accessibility_score",
    "vulnerability_score"
]

target = "impact_score"

X = df[feature_columns]
y = df[target]

# Remove missing values
data = pd.concat([X, y], axis=1).dropna()

X = data[feature_columns]
y = data[target]

# --------------------------------------------------
# Train-Test Split
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# --------------------------------------------------
# Models
# --------------------------------------------------

models = {

    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        random_state=42,
        objective="reg:squarederror"
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

# --------------------------------------------------
# Training and Evaluation
# --------------------------------------------------

results = []

for name, model in models.items():

    # Train model
    model.fit(X_train, y_train)

    # Predict on test data
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Regression accuracy
    # Prediction considered accurate if error <= 0.05
    accuracy = np.mean(
        np.abs(y_pred - y_test) <= 0.05
    ) * 100

    results.append([
        name,
        mse,
        mae,
        r2,
        accuracy
    ])

# --------------------------------------------------
# Results
# --------------------------------------------------

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "MSE",
        "MAE",
        "R2 Score",
        "Accuracy (%)"
    ]
)

print("\nMODEL PERFORMANCE")
print("=" * 70)
print(results_df.to_string(index=False))

# --------------------------------------------------
# Best Model
# --------------------------------------------------

best_model = results_df.loc[
    results_df["R2 Score"].idxmax()
]

print("\nBEST MODEL")
print("=" * 70)
print(best_model)


Training samples: 3635
Testing samples: 909

MODEL PERFORMANCE
            Model      MSE      MAE  R2 Score  Accuracy (%)
Linear Regression 0.000173 0.009737  0.883882     99.559956
    Random Forest 0.000069 0.005599  0.953978     99.889989
          XGBoost 0.000069 0.005640  0.953424     99.779978
Gradient Boosting 0.000083 0.005999  0.944505     99.779978

BEST MODEL
Model           Random Forest
MSE                  0.000069
MAE                  0.005599
R2 Score             0.953978
Accuracy (%)        99.889989
Name: 1, dtype: object
